In [10]:
import random_matrix.amplitude_matrix.isotropic_tmatrix as isotropic_tmatrix
from scipy.special import hankel1, spherical_jn, lpmv, h1vp, spherical_yn
import numpy as np
import math
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt
import pickle
import random_matrix.amplitude_matrix.scattering_geometry as scattering_geometry
import plotly.graph_objects as go
from scipy.integrate import quad
import debug_T_matrix


In [6]:
# Load the results
with open('Ts_full_sphere.pkl', 'rb') as f:
    results = pickle.load(f)

results.keys()
Ts_pnt001 = results['Ts'][4]
ns = results['n_stop']

In [7]:
ns

[3, 3, 4, 8, 10]

In [8]:
T_11 = T[0:35,0:35]
T_12 = T[0:35,35:70]
T_21 = T[35:70,0:35]
T_22 = T[35:70,35:70]
A = np.reshape(A,(A.shape[0],2,2))

NameError: name 'T' is not defined

In [ ]:
wavelength1 = 550e-9
k1 = (2 * np.pi) / wavelength1
m = 1.2  # Relative refractive Index
k2 = m * k1
Csca =0
for i in range(1,10):
    bn = debug_T_matrix.b_n_from_T(i,T_11)
    an = debug_T_matrix.a_n_from_T(i,T_22)
    Csca += np.abs(an)**2 + np.abs(bn)**2
Csca = ((2*np.pi)/k1**2)*Csca
print("Csca from T-matrix:",Csca)

Csca from T-matrix: 3.879444855498929e-15


In [18]:
wavelength1 = 500e-9
k1 = (2 * np.pi) / wavelength1
m = 1.2  # Relative refractive Index
k2 = m * k1
Csca =0
for i in range(1,20):
    bn = debug_T_matrix.b_n(i,1.2,2)
    an = debug_T_matrix.a_n(i,1.2,2)
    Csca += (2*i+1)*(np.abs(an)**2 + np.abs(bn)**2)
Csca = ((2*np.pi)/k1**2)*Csca
print("Csca from Mie theory:",Csca)

Csca from Mie theory: 1.9173040287686383e-14


In [5]:
print("fromT",debug_T_matrix.b_n_from_T(4,T_11))
print(debug_T_matrix.b_n(4,1.2,2))

NameError: name 'T_11' is not defined

In [20]:
# sampling incident field
n = 100
theta_i = 0 * np.ones((100))
phi_i = 0 * np.ones((101))
theta_grid_i, phi_grid_i = np.meshgrid(theta_i, phi_i)
# Incident field
ki_z = np.ravel(np.cos(theta_grid_i))
ki_x = np.ravel(np.sin(theta_grid_i) * np.cos(phi_grid_i))
ki_y = np.ravel(np.sin(theta_grid_i) * np.sin(phi_grid_i))
# sampling scattered field
theta_s = np.linspace(0, np.pi, n)
phi_s = np.linspace(0, 2 * np.pi, n + 1)
# theta = 0 * np.ones((100))
# phi = 0* np.ones((101))
theta_grid_s, phi_grid_s = np.meshgrid(theta_s, phi_s)
# Scattered field
ks_z = np.ravel(np.cos(theta_grid_s))
ks_x = np.ravel(np.sin(theta_grid_s) * np.cos(phi_grid_s))
ks_y = np.ravel(np.sin(theta_grid_s) * np.sin(phi_grid_s))
d_theta = np.pi / (n - 1)
d_phi = np.pi / (n)

In [32]:
input_pol = 0  # polarization angle of the incident field   
A = isotropic_tmatrix.scattering_amplitudes_from_T_v2(
    theta_grid_i, phi_grid_i, theta_grid_s, phi_grid_s, Ts_pnt001, k1, 10
)  
A = np.reshape(A, (A.shape[0], 2, 2))     
Ex = np.cos(input_pol)
Ey = np.sin(input_pol)

S11 = np.reshape(A[:,0,0], (n + 1, n))
S12 = np.reshape(A[:,0,1], (n + 1, n))
S21= np.reshape(A[:,1,0], (n + 1, n))
S22 = np.reshape(A[:,1,1], (n + 1, n))
Es_theta = S11 * Ex + S12 * Ey
Es_phi = S21 * Ex + S22 * Ey
Is = (np.abs(S11)**2 + np.abs(S21)**2) * np.sin(theta_grid_s) / k1**2
# Is = (np.abs(Es_theta) ** 2 + np.abs(Es_phi) ** 2) * np.sin(theta_grid_s) / k1**2
inner_integral = np.trapezoid(Is, phi_s, d_phi, axis=0)
Csca_ar= np.trapezoid(inner_integral, theta_s, d_theta)
print(f"Csca from Amplitude matrix with n-max = {10}:",Csca_ar)

Csca from Amplitude matrix with n-max = 10: 1.2350808975811663e-14


In [8]:

input_pol = 0 # 0 for x-polarized, pi/2 for y-polarized
A = A
Ex = np.cos(input_pol)
Ey = np.sin(input_pol)

S11 = np.reshape(A[:,0,0], (n + 1, n))
S12 = np.reshape(A[:,0,1], (n + 1, n))
S21= np.reshape(A[:,1,0], (n + 1, n))
S22 = np.reshape(A[:,1,1], (n + 1, n))
Es_theta = S11 * Ex + S12 * Ey
Es_phi = S21 * Ex + S22 * Ey
Is = (np.abs(Es_theta) ** 2 + np.abs(Es_phi) ** 2) * np.sin(theta_grid_s) / k1**2
inner_integral = np.trapezoid(Is, phi_s, d_phi, axis=0)
Csca_ar= np.trapezoid(inner_integral, theta_s, d_theta)
print("Csca from Amplitude matrix:",Csca_ar)

Csca from Amplitude matrix: 1.2181139697593389e-14


In [18]:
n_max = 5
T_sliced = debug_T_matrix.get_T_sliced(T,n_max)
A = isotropic_tmatrix.scattering_amplitudes_from_T_v2(
    theta_grid_i, phi_grid_i, theta_grid_s, phi_grid_s, T_sliced, k1, n_max
)  
A = np.reshape(A, (A.shape[0], 2, 2))     
Ex = np.cos(input_pol)
Ey = np.sin(input_pol)

S11 = np.reshape(A[:,0,0], (n + 1, n))
S12 = np.reshape(A[:,0,1], (n + 1, n))
S21= np.reshape(A[:,1,0], (n + 1, n))
S22 = np.reshape(A[:,1,1], (n + 1, n))
Es_theta = S11 * Ex + S12 * Ey
Es_phi = S21 * Ex + S22 * Ey
Is = (np.abs(S11)**2 + np.abs(S21)**2) * np.sin(theta_grid_s) / k1**2
# Is = (np.abs(Es_theta) ** 2 + np.abs(Es_phi) ** 2) * np.sin(theta_grid_s) / k1**2
inner_integral = np.trapezoid(Is, phi_s, d_phi, axis=0)
Csca_ar= np.trapezoid(inner_integral, theta_s, d_theta)
print(f"Csca from Amplitude matrix with n-max = {n_max}:",Csca_ar)

Csca from Amplitude matrix with n-max = 5: 1.2181139708717695e-14


In [19]:
import numpy as np
import matplotlib.pyplot as plt
import random_matrix.amplitude_matrix.isotropic_sphere as rm

wavelength = 400e-9
n = 100  # number of samples
rri = 1.2
m = np.reshape(rri * np.ones((n * (n + 1))), (n + 1, n))
k = (2 * np.pi) / wavelength

# sampling incident field
theta_i = 0 * np.ones((100))
phi_i = 0 * np.ones((101))
theta_grid_i, phi_grid_i = np.meshgrid(theta_i, phi_i)

# Incident field
ki_z = np.ravel(np.cos(theta_grid_i))
ki_x = np.ravel(np.sin(theta_grid_i) * np.cos(phi_grid_i))
ki_y = np.ravel(np.sin(theta_grid_i) * np.sin(phi_grid_i))

# sampling scattered field
theta = np.linspace(0, np.pi, n)
phi = np.linspace(0, 2 * np.pi, n + 1)
# theta = np.pi * np.ones((100))
# phi = 0* np.ones((101))
theta_grid, phi_grid = np.meshgrid(theta, phi)
# Scattered field
ks_z = np.ravel(np.cos(theta_grid))
ks_x = np.ravel(np.sin(theta_grid) * np.cos(phi_grid))
ks_y = np.ravel(np.sin(theta_grid) * np.sin(phi_grid))


size_param = np.linspace(0.01, 50, 1000)
C_scaT = np.zeros((1000))
radius1 = size_param / k
radius = 2 / k
A = rm.get_A(ki_x, ki_y, ki_z, ks_x, ks_y, ks_z, 2, rri)
d_theta = np.pi / (n - 1)
d_phi = np.pi / (n)
S2 = np.reshape(A[:, 0], (n + 1, n))
S3 = np.reshape(A[:, 1], (n + 1, n))
S4 = np.reshape(A[:, 2], (n + 1, n))
S1 = np.reshape(A[:, 3], (n + 1, n))

/home/sdutta/code/random-matrix/random_matrix/amplitude_matrix/scattering_geometry.py:30: RuntimeWarning: invalid value encountered in divide
  e_phi /= mod_kappa[..., xp.newaxis]
/home/sdutta/code/random-matrix/random_matrix/amplitude_matrix/scattering_geometry.py:44: RuntimeWarning: invalid value encountered in divide
  e_theta /= xp.sqrt(


In [26]:
np.isclose(S2,S11,rtol=1e-03,atol=1e-02)

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [False,  True,  True, ...,  True,  True,  True],
       ...,
       [False,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]], shape=(101, 100))